## **Apache Spark & Notebooks**
[Use Apache Spark in Microsoft Fabric](https://learn.microsoft.com/en-us/training/modules/use-apache-spark-work-files-lakehouse/)

### Drag & drop tables 
- Easily load data into data frame
- Quickly procure no-code visuals

In [ ]:
df = spark.sql("SELECT * FROM FinancialModeling.TransactionDescription_Silver LIMIT 1000")
display(df)

### Easily code in multiple languages (Magic command)
[Synapse Noteboks: Use multiple languages](https://learn.microsoft.com/en-us/azure/synapse-analytics/spark/apache-spark-development-using-notebooks#use-multiple-languages)

In [ ]:
%%sql

SELECT 
    gl.batch, gl.date, gl.account, gl.DR, gl.CR
    , td.Description , td.Anomaly_Indicator
FROM FinancialModeling.GeneralLedger_Silver gl 
LEFT JOIN FinancialModeling.TransactionDescription_Silver td
    ON ( td.Batch = gl.Batch)
LIMIT 10 

### Manipulate data across languages
- Example shows loading a SQL query into a dataframe for spark enrichment
- Inversely, one could create a database object in the Spark catalog for SQL enrichment (df.createOrReplaceTempView("Name"))

In [ ]:
df = spark.sql("""
SELECT 
    gl.batch, gl.date, gl.account, gl.DR, gl.CR
    , td.Description , td.Anomaly_Indicator
FROM FinancialModeling.GeneralLedger_Silver gl 
LEFT JOIN FinancialModeling.TransactionDescription_Silver td
    ON ( td.Batch = gl.Batch)
""")
df.show(10)


In [ ]:
from pyspark.sql.functions import when, col

# Assuming df is your existing DataFrame
df = df.withColumn("Anomaly_Indicator", when(col("Anomaly_Indicator") != "", 1).otherwise(0))

# Show the updated DataFrame
df.show(10)

### Save data back to Lakehouse

In [ ]:
df.write.format("delta").mode("overwrite").save(f"Tables/GeneralLedger_Gold")

## **DATA SCIENCE**
[Implement data science and machine learning for AI in Microsoft Fabric](https://learn.microsoft.com/en-us/training/paths/implement-data-science-machine-learning-fabric/)

### Data Wrangler
[Preprocess data with Data Wrangler in Microsoft Fabric](https://learn.microsoft.com/en-us/training/modules/preprocess-data-with-data-wrangler-microsoft-fabric/)
### Launch Data Wrangler for one-hot-encoding
*Note: Data Wrangler requires a dataframe*

In [ ]:
# Code generated by Data Wrangler for PySpark DataFrame

from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

def clean_data(df):
    # One-hot encode column: 'Description'
    # ⚠️ This was generated to match the original pandas logic but may have performance issues.
    
    def one_hot_encode_col(df, key):
        indexer = StringIndexer(inputCol=key, outputCol='%s_numeric' % str(key), handleInvalid='keep')
        indexer_fitted = indexer.fit(df)
        df_indexed = indexer_fitted.transform(df)
        encoder = OneHotEncoder(inputCols=['%s_numeric' % str(key)], outputCols=['%s_onehot' % str(key)], dropLast=False)
        df_onehot = encoder.fit(df_indexed).transform(df_indexed)
        df_col_onehot = df_onehot.select('*', vector_to_array('%s_onehot' % str(key)).alias('%s_col_onehot' % str(key)))
        labels = sorted(indexer_fitted.labels)
        cols_expanded = [(F.col('%s_col_onehot' % str(key))[i].alias('%s_%s' % (str(key), labels[i]))) for i in range(len(labels))]
        df = df_col_onehot.select(*df.columns, *cols_expanded)
        df = df.drop(key)
        return df
    
    df = one_hot_encode_col(df, 'Description')
    return df

df_clean = clean_data(df)
display(df_clean)

#### Explore data

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Convert the PySpark DataFrame to Pandas
pandas_df = df_clean.toPandas()

# Convert 'DR' and 'CR' columns to numeric type (float)
pandas_df['DR'] = pd.to_numeric(pandas_df['DR'], errors='coerce')
pandas_df['CR'] = pd.to_numeric(pandas_df['CR'], errors='coerce')

# Specify the columns to be dropped
columns_to_drop = ['batch']  # Replace with actual column names

# Drop the specified columns
pandas_df = pandas_df.drop(columns=columns_to_drop, axis=1)

# Select only numeric columns for correlation calculation
numeric_cols = pandas_df.select_dtypes(include=[np.number])

# Calculate the correlation matrix and round to two decimal places
corr = numeric_cols.corr().round(2)

# Create a heatmap with smaller annotations
sns.heatmap(corr, annot=True, annot_kws={"size": 5}, cmap='coolwarm')
plt.show()
